# AVA Spatial Agent Integration

Combines DroneKit control with GeoJSON spatial queries.
The drone can now find and navigate to conservation targets.

In [1]:
!pip install -q agno dronekit pymavlink pydantic


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = input('OpenRouter API Key: ')

print('API key set.' if os.environ.get('OPENROUTER_API_KEY') else 'No API key!')

API key set.


In [3]:
from dronekit import connect, VehicleMode
import time

CONNECTION_STRING = 'tcp:127.0.0.1:5763'

print('Connecting to vehicle...')
vehicle = connect(CONNECTION_STRING, wait_ready=True, baud=57600, rate=60)
print(f'Connected!')
print(f'  Mode:     {vehicle.mode.name}')
print(f'  GPS:      {vehicle.gps_0}')
print(f'  Position: {vehicle.location.global_frame.lat}, {vehicle.location.global_frame.lon}')

Connecting to vehicle...
Connected!
  Mode:     STABILIZE
  GPS:      GPSInfo:fix=6,num_sat=10
  Position: -35.3633517, 149.1652414


## Import Spatial and Drone Tools

In [4]:
from agno.tools import tool
from agents.spatial_agent import spatial_tools

# Basic drone control tools
@tool
def get_current_position() -> str:
    """
    Get current GPS coordinates of the drone.
    """
    lat = vehicle.location.global_frame.lat
    lon = vehicle.location.global_frame.lon
    alt = vehicle.location.global_relative_frame.alt
    return f"Lat: {lat}, Lon: {lon}, Alt: {alt}m"

@tool
def go_to_coords(lat: float, lon: float, alt: float = None) -> str:
    """
    Fly to GPS coordinates.
    Args:
        lat: Destination latitude
        lon: Destination longitude
        alt: Destination altitude in meters (optional)
    """
    from dronekit import LocationGlobalRelative
    
    if alt is None:
        alt = vehicle.location.global_relative_frame.alt
    
    target = LocationGlobalRelative(lat, lon, alt)
    vehicle.simple_goto(target)
    
    return f"Flying to {lat}, {lon} at {alt}m"

# Combine all tools
all_tools = spatial_tools + [get_current_position, go_to_coords]

print(f"Loaded {len(all_tools)} tools:")
for t in all_tools:
    print(f"  - {t.name}")

Loaded 5 tools:
  - get_nearest_target
  - get_target_by_name
  - list_all_targets
  - get_current_position
  - go_to_coords


## Build Combined Agent

In [5]:
from agno.agent import Agent  # Note: agno.agent not just agno
from agno.models.openrouter import OpenRouter

SYSTEM_PROMPT = """
You are AVA - an autonomous assistant for conservation drone operations.

You can:
- Query conservation targets from the GeoJSON database
- Find nearest targets to the drone's position
- Navigate the drone to target locations
- Provide target metadata and observation history

When asked to find or visit a target:
1. Get current drone position
2. Query for nearest target (or lookup by name)
3. Navigate to target coordinates
4. Report target details

Be concise and operational in tone.
"""

agent = Agent(
    name='AVA_Combined',
    model=OpenRouter(
        id='qwen/qwen3-vl-30b-a3b-thinking',  # ← NEW MODEL
        api_key=os.environ['OPENROUTER_API_KEY'],
    ),
    instructions=SYSTEM_PROMPT,
    tools=all_tools,
    add_history_to_context=True,
    debug_mode=True
)

print('Agent ready.')

Agent ready.


## Test Queries

In [6]:
# Test 1: Find nearest target to current position
result = agent.run("What's the nearest conservation target to my current position?")
print(result.content)

WARNING  add_history_to_context is True, but no database has been assigned to the agent. History will not be added 
         to the context.

DEBUG ****************** Agent ID: ava-combined ******************

DEBUG Creating new AgentSession: 061c8244-2452-437b-a45f-0dd5007e08c4

DEBUG ** Agent Run Start: 8e3b1fe2-3a56-4c43-b8ce-b9f0b69a005a ***

DEBUG Processing tools for model

...

DEBUG ************************  METRICS  *************************

DEBUG ----------------- OpenRouter Response End ------------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 8e3b1fe2-3a56-4c43-b8ce-b9f0b69a005a ****

Nearest target: Alpine Creek Water Source (ID: target_002), 13638.9km away. Coordinates: 40.018°N, 105.282°W. Priority: medium. Ready to navigate?


In [7]:
# Test 2: Lookup specific target
result = agent.run("Tell me about Rocky Ridge Wildlife Habitat")
print(result.content)

WARNING  add_history_to_context is True, but no database has been assigned to the agent. History will not be added 
         to the context.

DEBUG ***** Session ID: 061c8244-2452-437b-a45f-0dd5007e08c4 *****

DEBUG ****************** Agent ID: ava-combined ******************

DEBUG Creating new AgentSession: 061c8244-2452-437b-a45f-0dd5007e08c4

DEBUG ** Agent Run Start: a693ecde-2e87-4373-ad52-c834572113e3 ***

...

DEBUG ************************  METRICS  *************************

DEBUG ----------------- OpenRouter Response End ------------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: a693ecde-2e87-4373-ad52-c834572113e3 ****

No target named "Rocky Ridge Wildlife Habitat" found. Closest matches:  
- **North Ridge Eagle Nest** (ID: target_001) - Visited 1x, 1 observation  
- **East Valley Meadow** (ID: target_004) - Visited 2x, 2 observations  

Both are high-priority wildlife habitats. Would you like to navigate to either?


In [8]:
# Test 3: Navigate to nearest target
result = agent.run("Take me to the nearest high priority target")
print(result.content)

WARNING  add_history_to_context is True, but no database has been assigned to the agent. History will not be added 
         to the context.

DEBUG ***** Session ID: 061c8244-2452-437b-a45f-0dd5007e08c4 *****

DEBUG ****************** Agent ID: ava-combined ******************

DEBUG Creating new AgentSession: 061c8244-2452-437b-a45f-0dd5007e08c4

DEBUG ** Agent Run Start: 08954212-f966-439d-b642-3359b509aa2a ***

...

DEBUG ************************  METRICS  *************************

DEBUG ----------------- OpenRouter Response End ------------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 08954212-f966-439d-b642-3359b509aa2a ****

Flying to nearest high-priority target: North Ridge Eagle Nest (ID: target_001) at 40.015°N, 105.2705°W. Coordinates confirmed. Proceeding to destination.


In [9]:
# Test 4: List unvisited targets
result = agent.run("Show me all targets that haven't been visited yet")
print(result.content)

WARNING  add_history_to_context is True, but no database has been assigned to the agent. History will not be added 
         to the context.

DEBUG ***** Session ID: 061c8244-2452-437b-a45f-0dd5007e08c4 *****

DEBUG ****************** Agent ID: ava-combined ******************

DEBUG Creating new AgentSession: 061c8244-2452-437b-a45f-0dd5007e08c4

DEBUG ** Agent Run Start: 6bb882bd-78c3-4ee4-8e82-eef733a4ec09 ***

...

DEBUG ----------------- OpenRouter Response End ------------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 6bb882bd-78c3-4ee4-8e82-eef733a4ec09 ****

Two unvisited targets available:
- **Alpine Creek Water Source** (water_source, medium priority)  
  Coordinates: 40.018° N, 105.282° W | 0 visits
- **South Slope Aspen Grove** (vegetation, medium priority)  
  Coordinates: 40.009° N, 105.268° W | 0 visits

Ready to navigate to either location. Confirm target selection.


ERROR:dronekit.mavlink:Exception in MAVLink input loop
Traceback (most recent call last):
  File "c:\Users\kavir\dronekit3\Lib\site-packages\dronekit\mavlink.py", line 211, in mavlink_thread_in
    fn(self)
  File "c:\Users\kavir\dronekit3\Lib\site-packages\dronekit\__init__.py", line 1370, in listener
    raise APIException('No heartbeat in %s seconds, aborting.' %
dronekit.APIException: No heartbeat in 30 seconds, aborting.


## Interactive Mode

In [10]:
print('AVA Spatial Agent - Interactive Mode')
print('Type commands. Type "exit" to quit.\n')

while True:
    user_input = input('\nCommand: ')
    
    if user_input.lower().strip() == 'exit':
        break
    
    if not user_input.strip():
        continue
    
    try:
        result = agent.run(user_input)
        print(f'\nResponse: {result.content}')
    except Exception as e:
        print(f'\nERROR: {e}')

print('Session ended.')

AVA Spatial Agent - Interactive Mode
Type commands. Type "exit" to quit.



WARNING  add_history_to_context is True, but no database has been assigned to the agent. History will not be added 
         to the context.

DEBUG ***** Session ID: 061c8244-2452-437b-a45f-0dd5007e08c4 *****

DEBUG ****************** Agent ID: ava-combined ******************

DEBUG Creating new AgentSession: 061c8244-2452-437b-a45f-0dd5007e08c4

...

DEBUG ************************  METRICS  *************************

DEBUG ----------------- OpenRouter Response End ------------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 69bf9f65-8372-4937-bf5f-5b5447df82d7 ****


Response: Drone position: 40.018°N, 105.282°W (current altitude: 2380m)

**Route Plan:**
1. Navigate to **Alpine Creek Water Source** (target_002) - 13638.9km from current position
2. Then proceed to **South Slope Aspen Grove** (target_005) - 1.85km from first target

Execute flight path to target_002 using go_to_coords. After successful visit, initiate next waypoint to target_005. All targets require full environmental scanning upon arrival. Confirm readiness to proceed?
Session ended.


In [ ]:
vehicle.close()
print('Vehicle connection closed.')